# FeynmanEngine for an LHC Experimentalist

**Audience**: experimentalist or phenomenologist who wants quick, defensible LO/NLO predictions for LHC processes — no need to spin up MadGraph or NNLOJET for a back-of-envelope check.

**Goal**: predict cross-sections + differential distributions for the major LHC channels and compare to ATLAS/CMS measurements.

**Prerequisite**: `pip install feynman-engine && feynman setup` (the full setup brings in LHAPDF + CT18LO and OpenLoops 2 — both needed for percent-level hadronic accuracy and generic NLO QCD virtuals).

---

**Outline:**
1. The LHC σ benchmark grid at √s = 13 TeV (engine vs literature)
2. Drell-Yan: σ + K-factor + dσ/dM_ll spectrum across the Z pole
3. Top pair production with full massive-top kinematics
4. Higgs production: ggF, VBF, ZH — with the right K-factors
5. EW Sudakov suppression at high-pT tails
6. The trust system: when the engine refuses to give you a number

**For users without LHAPDF/OpenLoops**: skip cells will tell you and continue gracefully.

## 1. LHC benchmark grid at 13 TeV

Pick a process, run the calculation, compare to the literature. The engine self-reports `trust_level` and `accuracy_caveat` so you know when the number is solid vs. an approximation.

In [ ]:
from feynman_engine.amplitudes.hadronic import hadronic_cross_section

lhc_processes = [
    # (process,           theory, expected_pb, reference)
    ("p p -> t t~",       "QCD",  830.0,  "ATLAS+CMS measured σ_tt̄ ≈ 830 pb"),
    ("p p -> H",          "QCD",  48.6,   "LHC HWG YR4 NNLO+NNLL ggH (engine returns LO×K=1.7)"),
    ("p p -> H j j",      "QCD",  3.78,   "LHC HWG YR4 VBF (engine calibrated)"),
    ("p p -> Z H",        "EW",   0.5,    "LHC HWG YR4 qq̄→ZH"),
    ("p p -> Z Z",        "EW",   10.0,   "ATLAS qq̄→ZZ"),
    ("p p -> W+ W-",      "EW",   50.0,   "ATLAS+CMS (engine is t-channel only — see trust caveat)"),
]

print(f"  {'Process':<18s} σ_engine [pb]   σ_ref [pb]    deviation   trust")
print("  " + "-" * 80)
for proc, theory, ref, label in lhc_processes:
    r = hadronic_cross_section(proc, sqrt_s=13000.0, theory=theory)
    if r.get("supported"):
        sigma = r["sigma_pb"]
        dev = (sigma / ref - 1) * 100
        trust = r.get("trust_level", "?")
        print(f"  {proc:<18s} {sigma:>13.2f}   {ref:>10.2f}   {dev:+6.1f}%   {trust}")
    else:
        print(f"  {proc:<18s} (unavailable: {r.get('error', '?')[:40]})")

## 2. Drell-Yan — σ, K-factor, and dσ/dM_ll

The engine has a specialised analytic γ+Z Drell-Yan path that's fast (<1 s) and matches the LHC measured σ within ~25% (built-in PDF) or ~5% (LHAPDF + CT18LO).

**K-factor**: Catani-Seymour subtraction with HMNvN coefficient functions reproduces Anastasiou et al.'s `K_NLO = 1.21` to 1.6%.

In [ ]:
for sqrts, ref_pb in [(7000, 1100), (8000, 1300), (13000, 2000), (14000, 2200)]:
    rl = hadronic_cross_section("p p -> e+ e-", sqrt_s=float(sqrts), theory="QCD", order="LO")
    rn = hadronic_cross_section("p p -> e+ e-", sqrt_s=float(sqrts), theory="QCD", order="NLO")
    K = rn.get("k_factor_applied") or rn.get("k_factor")
    print(f"  √s = {sqrts:>5} GeV:  σ_LO = {rl['sigma_pb']:>6.0f} pb   σ_NLO = {rn['sigma_pb']:>6.0f} pb   K = {K:.3f}   (LHC ~{ref_pb} pb)")

In [ ]:
# Differential dσ/dM_ll across the Z resonance
import numpy as np
import matplotlib.pyplot as plt

try:
    from feynman_engine.amplitudes.nlo_general import hadronic_nlo_dy_differential_v26
    r = hadronic_nlo_dy_differential_v26(sqrt_s_gev=13000.0, m_ll_min=70.0, m_ll_max=110.0, n_bins=8)
    centers = np.array(r["bin_centers"])
    lo = np.array(r["dsigma_dM_lo_pb_per_gev"])
    nlo = np.array(r["dsigma_dM_nlo_pb_per_gev"])
    K = np.array(r["k_per_bin"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].step(centers, lo, where="mid", label="LO", color="steelblue")
    axes[0].step(centers, nlo, where="mid", label="NLO", color="firebrick")
    axes[0].set_xlabel(r"$M_{\ell\ell}$ [GeV]")
    axes[0].set_ylabel(r"$d\sigma/dM_{\ell\ell}$ [pb/GeV]")
    axes[0].set_title("DY at LHC 13 TeV across the Z peak")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].step(centers, K, where="mid", color="darkgreen")
    axes[1].axhline(1.21, color="gray", ls="--", label="YR4 K = 1.21")
    axes[1].set_xlabel(r"$M_{\ell\ell}$ [GeV]")
    axes[1].set_ylabel("K-factor per bin")
    axes[1].set_title("K-factor stability across the bins")
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
except Exception as exc:
    print(f"Skipping (LHAPDF or grid build unavailable): {exc}")

## 3. Top-pair production — gg fusion + qq̄ annihilation

Massive top kinematics (Combridge formulas) on both gg and qq̄ channels. At 13 TeV gluon fusion dominates (~85%); at lower energy qq̄ wins.

In [ ]:
for sqrts in [7000, 8000, 13000, 14000]:
    r = hadronic_cross_section("p p -> t t~", sqrt_s=float(sqrts), theory="QCD", order="NLO")
    K = r.get("k_factor_applied") or r.get("k_factor", 1.0)
    print(f"  √s = {sqrts:>5} GeV:  σ_NLO(pp→tt̄) = {r['sigma_pb']:>5.0f} pb  (K = {K:.2f})")

## 4. Higgs production — ggF, VBF, ZH

Three channels with very different kinematics. The engine has specialised analytic paths for each:
- **ggF**: heavy-top effective theory + NLO K = 1.7
- **VBF (pp → H jj)**: calibrated to LHC HWG YR4 reference at √s = 13 TeV
- **ZH (Higgsstrahlung)**: 5 quark flavour qq̄ → ZH channels via Hagiwara-Zeppenfeld

In [ ]:
for proc, theory, label in [
    ("p p -> H",      "QCD", "gluon fusion (ggF)"),
    ("p p -> H j j",  "QCD", "vector-boson fusion (VBF)"),
    ("p p -> Z H",    "EW",  "Higgsstrahlung (ZH)"),
]:
    for order in ("LO", "NLO"):
        r = hadronic_cross_section(proc, sqrt_s=13000.0, theory=theory, order=order)
        sigma = r.get("sigma_pb", "?")
        K = r.get("k_factor_applied") or r.get("k_factor")
        K_str = f" K={K:.2f}" if K and order == "NLO" else ""
        print(f"  σ_{order:<3s}({label:<28s}) = {sigma:>6.2f} pb{K_str}")

## 5. EW Sudakov at high-pT tails

At LHC energies $\sqrt{s} \gg M_W$, the dominant electroweak NLO correction is the Sudakov double log:
$$\delta_{EW} = -\frac{\alpha}{4\pi \sin^2\theta_W} \sum T_{eff}^2 \big(L^2 + 3L\big), \quad L = \log(s/M_W^2)$$

For LHC analyses with $M_{ll}$ or $p_T$ in the TeV range, this is the leading EW correction — **always negative** and **growing with energy**.

In [ ]:
from feynman_engine.amplitudes.nlo_ew_general import ew_nlo_sudakov_kfactor

energies = np.geomspace(200, 5000, 25)
K_vals = [ew_nlo_sudakov_kfactor("e+ e- -> mu+ mu-", float(e)).k_factor for e in energies]

plt.figure(figsize=(8, 4.5))
plt.semilogx(energies, K_vals, "o-", color="purple", lw=2)
plt.axhline(1.0, color="gray", ls="--", alpha=0.5)
plt.xlabel(r"$\sqrt{s}$ [GeV]")
plt.ylabel(r"$K_{EW}^{Sudakov}$")
plt.title(r"EW Sudakov suppression for $e^+e^- \to \mu^+\mu^-$ — universal LL+NLL")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

for e, K in zip(energies[::5], K_vals[::5]):
    print(f"  √s = {e:>6.0f} GeV: K_EW = {K:.3f}  ({(K-1)*100:+.1f}%)")

## 6. The trust system — when the engine refuses

Trying to compute σ for a process the engine knows it can't answer reliably returns **HTTP 422** with a structured `block_reason` and `workaround`. You never get a wrong number masquerading as a right one.

Example: requesting NLO for an unregistered QCD process without OpenLoops installed.

In [ ]:
from fastapi.testclient import TestClient
from feynman_engine.api.app import app

client = TestClient(app)
r = client.get("/api/amplitude/cross-section", params={
    "process": "u u~ -> c c~", "theory": "QCD", "sqrt_s": 91.0, "order": "NLO",
})
print(f"HTTP {r.status_code}")
if r.status_code == 422:
    d = r.json()["detail"]
    print(f"  trust_level: {d['trust_level']}")
    print(f"  block_reason: {d['block_reason']}")
    print(f"  workaround:   {d['workaround']}")

## What's next for an LHC analysis

- **`/api/amplitude/differential-distribution`** — bin-based dσ/dX for cosθ, pT, η, y, M_inv, M_ll, ΔR.
- **OpenLoops virtual-K**: install `feynman install-process pptt` for ttbar, `pphjj` for VBF Higgs, etc.
- **PDF variation**: pass `pdf_name="NNPDF40_lo_as_01180"` after `feynman install-pdf-set NNPDF40_lo_as_01180`.
- **For NNLO precision predictions** (V+jet, V+V, single-Higgs spectra): use NNLOJET or MCFM. This engine handles LO + NLO and is the right tool for fast cross-checks and coverage of the trust gates.